In [1]:
import pandas as pd

df = pd.read_csv("tft_test_forecasts.csv")
print(df.shape)
print(df.columns.tolist())
df.head()


(3237, 18)
['store_nbr', 'item_nbr', 'target_start_date', 'target_end_date', 'actual_day_1', 'pred_day_1', 'actual_day_2', 'pred_day_2', 'actual_day_3', 'pred_day_3', 'actual_day_4', 'pred_day_4', 'actual_day_5', 'pred_day_5', 'actual_day_6', 'pred_day_6', 'actual_day_7', 'pred_day_7']


,store_nbr,item_nbr,target_start_date,target_end_date,actual_day_1,pred_day_1,actual_day_2,pred_day_2,actual_day_3,pred_day_3,actual_day_4,pred_day_4,actual_day_5,pred_day_5,actual_day_6,pred_day_6,actual_day_7,pred_day_7
0,1,159156,2017-08-01,2017-08-07,20.454,18.492943,8.406,21.011591,9.551,13.047191,27.484,26.823202,6.503,7.560717,0.000,4.054430,2.089,11.700099
1,1,159156,2017-08-02,2017-08-08,8.406,17.059920,9.551,11.777991,27.484,26.013447,6.503,7.753114,0.000,4.286967,2.089,11.556550,3.624,11.512097
2,1,159156,2017-08-03,2017-08-09,9.551,10.930450,27.484,23.679730,6.503,7.082459,0.000,3.977376,2.089,10.358905,3.624,10.284374,5.997,12.730545
3,1,159156,2017-08-04,2017-08-10,27.484,22.707241,6.503,7.787842,0.000,4.355536,2.089,10.935470,3.624,10.803349,5.997,13.184616,28.876,9.223534
4,1,159156,2017-08-05,2017-08-11,6.503,9.156804,0.000,4.844530,2.089,13.203774,3.624,13.059927,5.997,15.670193,28.876,11.108266,7.943,9.605672


In [2]:
# Reshape wide -> long format
rows = []
for day in range(1, 8):
    temp = df[['store_nbr', 'item_nbr', 'target_start_date', 
               f'actual_day_{day}', f'pred_day_{day}']].copy()
    temp = temp.rename(columns={f'actual_day_{day}': 'y_true', 
                                  f'pred_day_{day}': 'y_pred'})
    temp['horizon_day'] = day
    rows.append(temp)

df_long = pd.concat(rows, ignore_index=True)
df_long['error'] = df_long['y_true'] - df_long['y_pred']
df_long['abs_error'] = df_long['error'].abs()
df_long['sq_error'] = df_long['error'] ** 2

print(df_long.shape)
df_long.head()

(22659, 9)


,store_nbr,item_nbr,target_start_date,y_true,y_pred,horizon_day,error,abs_error,sq_error
0,1,159156,2017-08-01,20.454,18.492943,1,1.961057,1.961057,3.845745
1,1,159156,2017-08-02,8.406,17.059920,1,-8.653920,8.653920,74.890331
2,1,159156,2017-08-03,9.551,10.930450,1,-1.379450,1.379450,1.902882
3,1,159156,2017-08-04,27.484,22.707241,1,4.776759,4.776759,22.817427
4,1,159156,2017-08-05,6.503,9.156804,1,-2.653804,2.653804,7.042676


In [3]:
import numpy as np

mae = df_long['abs_error'].mean()
rmse = np.sqrt(df_long['sq_error'].mean())
ratio = rmse / mae

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"RMSE/MAE ratio: {ratio:.3f}")


MAE:  12.387
RMSE: 29.542
RMSE/MAE ratio: 2.385


In [4]:
df_sorted = df_long.sort_values('sq_error', ascending=False).reset_index(drop=True)
total_sq_error = df_long['sq_error'].sum()

for pct in [0.01, 0.05, 0.10]:
    n = int(len(df_sorted) * pct)
    top_sq_error = df_sorted.iloc[:n]['sq_error'].sum()
    share = top_sq_error / total_sq_error * 100
    print(f"Top {pct*100:.0f}% of rows ({n} rows) contribute {share:.1f}% of total squared error")
    

Top 1% of rows (226 rows) contribute 62.1% of total squared error
Top 5% of rows (1132 rows) contribute 84.3% of total squared error
Top 10% of rows (2265 rows) contribute 91.7% of total squared error


In [5]:
pd.set_option('display.max_columns', None)
df_sorted.head(20)[['store_nbr', 'item_nbr', 'target_start_date', 'horizon_day', 'y_true', 'y_pred', 'error', 'sq_error']]

,store_nbr,item_nbr,target_start_date,horizon_day,y_true,y_pred,error,sq_error
0,39,164647,2017-08-01,1,737.000,18.221998,718.778002,516641.816159
1,39,1167614,2017-08-01,7,1654.000,989.560600,664.439400,441479.716272
2,39,581078,2017-08-03,1,653.000,38.672382,614.327618,377398.422238
3,39,581078,2017-08-02,2,653.000,39.002148,613.997852,376993.362261
4,39,581078,2017-08-01,3,653.000,39.534904,613.465096,376339.424010
5,2,582864,2017-08-04,7,627.889,134.722470,493.166530,243213.226312
6,2,582864,2017-08-06,5,627.889,138.515370,489.373630,239486.549739
7,2,582864,2017-08-05,6,627.889,138.674990,489.214010,239330.347580
8,2,582864,2017-08-07,4,627.889,138.932370,488.956630,239078.586021
9,2,582864,2017-08-08,3,627.889,139.569100,488.319900,238456.324736


In [6]:
# How many UNIQUE store-item pairs are driving the top 1% errors?
top1pct = df_sorted.head(226)
print("Unique store-item pairs in top 1% worst errors:", 
      top1pct[['store_nbr','item_nbr']].drop_duplicates().shape[0])
print(top1pct[['store_nbr','item_nbr']].drop_duplicates())

# How many unique CALENDAR dates are actually involved (accounting for overlap)?
# derive actual calendar date = target_start_date + (horizon_day - 1)
df_long['target_start_date'] = pd.to_datetime(df_long['target_start_date'])
df_long['calendar_date'] = df_long['target_start_date'] + pd.to_timedelta(df_long['horizon_day'] - 1, unit='D')

top1pct_dates = df_sorted.head(226).copy()
top1pct_dates['target_start_date'] = pd.to_datetime(top1pct_dates['target_start_date'])
top1pct_dates['calendar_date'] = top1pct_dates['target_start_date'] + pd.to_timedelta(top1pct_dates['horizon_day'] - 1, unit='D')
print("\nUnique (store, item, calendar_date) combos in top 1%:",
      top1pct_dates[['store_nbr','item_nbr','calendar_date']].drop_duplicates().shape[0])

Unique store-item pairs in top 1% worst errors: 20
     store_nbr  item_nbr
0           39    164647
1           39   1167614
2           39    581078
5            2    582864
16          47   1047679
19          44   1047679
25           8   1047679
28          11   1047679
37          11    584126
43          38   1004550
49          49   1392260
58           3   1047679
59          47   1730015
63          48    159156
83          20   1463992
136         46   1047679
138         11    807493
188         25   1146496
193         48    903285
205         47    903285

Unique (store, item, calendar_date) combos in top 1%: 69


In [7]:
pairs_meta = pd.read_csv("selected_pairs.csv")
print(pairs_meta.columns.tolist())

bad_pairs = top1pct[['store_nbr','item_nbr']].drop_duplicates()
bad_pairs_meta = bad_pairs.merge(pairs_meta, on=['store_nbr','item_nbr'], how='left')
print(bad_pairs_meta)

['store_nbr', 'item_nbr', 'observation_count', 'total_sales', 'first_date', 'last_date', 'calendar_days', 'coverage', 'average_sales', 'family', 'class', 'perishable', 'log_total_sales', 'history_score', 'coverage_score', 'activity_score', 'base_score', 'scarcity_bonus', 'priority_score', 'demand_bin', 'coverage_bin', 'history_bin']
    store_nbr  item_nbr  observation_count    total_sales  first_date  \
0          39    164647               1660   56470.000000  2013-01-02   
1          39   1167614               1438  468546.000000  2013-08-10   
2          39    581078               1661   40561.000000  2013-01-02   
3           2    582864               1664  224271.859375  2013-01-02   
4          47   1047679               1664  318344.000000  2013-01-02   
5          44   1047679               1664  826437.000000  2013-01-02   
6           8   1047679               1664  378332.000000  2013-01-02   
7          11   1047679               1662  245858.000000  2013-01-02   
8       

In [8]:
bad_pairs_list = bad_pairs_meta[['store_nbr','item_nbr']]
mask = df_long.set_index(['store_nbr','item_nbr']).index.isin(
    bad_pairs_list.set_index(['store_nbr','item_nbr']).index
)
df_bad = df_long[mask]

print("Rows belonging to these 20 pairs:", df_bad.shape[0], 
      f"({df_bad.shape[0]/len(df_long)*100:.1f}% of all rows)")
print("Their share of total squared error:", 
      f"{df_bad['sq_error'].sum()/df_long['sq_error'].sum()*100:.1f}%")
print("\nMean y_true:", df_bad['y_true'].mean())
print("Mean y_pred:", df_bad['y_pred'].mean())
print("Mean error (true - pred):", df_bad['error'].mean())
print("% of rows where model UNDERpredicts (error > 0):", (df_bad['error'] > 0).mean()*100)

Rows belonging to these 20 pairs: 1204 (5.3% of all rows)
Their share of total squared error: 74.5%

Mean y_true: 190.54168521594684
Mean y_pred: 181.94372157027408
Mean error (true - pred): 8.597963645672758
% of rows where model UNDERpredicts (error > 0): 47.25913621262458


In [9]:
day_breakdown = df_bad.groupby('horizon_day').agg(
    mae=('abs_error','mean'),
    rmse=('sq_error', lambda x: np.sqrt(x.mean())),
    n=('abs_error','count')
)
day_breakdown['ratio'] = day_breakdown['rmse']/day_breakdown['mae']
print(day_breakdown)

                   mae        rmse    n     ratio
horizon_day                                      
1            76.124394  126.814971  172  1.665891
2            75.922673  121.036426  172  1.594207
3            73.549422  113.390387  172  1.541690
4            67.222501   99.309698  172  1.477328
5            66.985357   98.149279  172  1.465235
6            70.316066  104.738401  172  1.489537
7            67.299703  107.531681  172  1.597803
